<a href="https://colab.research.google.com/github/prathibhaogirala-creator/Pallavi-Ogirala/blob/main/Artifacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Cayley-Hamilton Verifier</title>
    <link rel="stylesheet" href="style.css">
</head>

<body>

<div class="container">
    <h1>Cayley-Hamilton Verifier</h1>
    <p>Verify whether a matrix satisfies the Cayley-Hamilton theorem.</p>

    <label>Select Matrix Size:</label>
    <select id="matrixSize" onchange="createMatrix()">
        <option value="2">2 × 2</option>
        <option value="3">3 × 3</option>
    </select>

    <div id="matrixContainer"></div>

    <button onclick="verifyMatrix()">Verify Matrix</button>
    <button onclick="clearMatrix()">Clear</button>

    <div id="result"></div>
</div>

<script src="script.js"></script>

</body>
</html>

In [ ]:
body {
    font-family: Arial, sans-serif;
    background: #f4f6f8;
    text-align: center;
    margin: 0;
    padding: 30px;
}

.container {
    max-width: 700px;
    margin: auto;
    background: white;
    padding: 30px;
    border-radius: 15px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.15);
}

h1 {
    margin-bottom: 10px;
}

select {
    padding: 10px;
    margin: 15px;
    font-size: 16px;
}

.matrix {
    display: grid;
    gap: 10px;
    justify-content: center;
    margin: 20px;
}

.matrix input {
    width: 60px;
    height: 40px;
    text-align: center;
    font-size: 18px;
    border: 1px solid #aaa;
    border-radius: 6px;
}

button {
    padding: 12px 20px;
    margin: 8px;
    border: none;
    border-radius: 8px;
    cursor: pointer;
    font-size: 16px;
}

#result {
    margin-top: 20px;
    padding: 15px;
    border-radius: 8px;
    font-size: 17px;
}

In [ ]:
function createMatrix() {
    const n = parseInt(document.getElementById("matrixSize").value);
    const container = document.getElementById("matrixContainer");

    container.innerHTML = "";

    const matrix = document.createElement("div");
    matrix.className = "matrix";
    matrix.style.gridTemplateColumns = `repeat(${n}, 60px)`;

    for (let i = 0; i < n * n; i++) {
        const input = document.createElement("input");
        input.type = "number";
        input.value = "0";
        input.id = `a${i}`;
        matrix.appendChild(input);
    }

    container.appendChild(matrix);
}

function getMatrix() {
    const n = parseInt(document.getElementById("matrixSize").value);
    let matrix = [];

    for (let i = 0; i < n; i++) {
        let row = [];

        for (let j = 0; j < n; j++) {
            row.push(
                Number(document.getElementById(`a${i * n + j}`).value)
            );
        }

        matrix.push(row);
    }

    return matrix;
}

function determinant(A) {
    const n = A.length;

    if (n === 1) return A[0][0];

    if (n === 2) {
        return A[0][0] * A[1][1] -
               A[0][1] * A[1][0];
    }

    let det = 0;

    for (let j = 0; j < n; j++) {
        let minor = A.slice(1).map(row =>
            row.filter((_, col) => col !== j)
        );

        det += Math.pow(-1, j) * A[0][j] * determinant(minor);
    }

    return det;
}

function multiply(A, B) {
    const n = A.length;
    let C = Array.from({length: n}, () => Array(n).fill(0));

    for (let i = 0; i < n; i++) {
        for (let j = 0; j < n; j++) {
            for (let k = 0; k < n; k++) {
                C[i][j] += A[i][k] * B[k][j];
            }
        }
    }

    return C;
}

function identity(n) {
    return Array.from({length: n}, (_, i) =>
        Array.from({length: n}, (_, j) => i === j ? 1 : 0)
    );
}

function subtract(A, B) {
    return A.map((row, i) =>
        row.map((value, j) => value - B[i][j])
    );
}

function scalarMultiply(A, k) {
    return A.map(row => row.map(value => value * k));
}

function verifyMatrix() {
    const A = getMatrix();
    const n = A.length;

    let trace = A.reduce((sum, row, i) => sum + row[i], 0);
    let det = determinant(A);

    let result;

    if (n === 2) {
        // Characteristic equation:
        // λ² - trace(A)λ + det(A) = 0

        let A2 = multiply(A, A);

        let polynomial = subtract(
            subtract(
                A2,
                scalarMultiply(A, trace)
            ),
            scalarMultiply(identity(2), -det)
        );

        result = polynomial;
    }

    if (n === 3) {
        // For 3x3 matrices:
        // p(A) = A³ - c1 A² + c2 A - c3 I

        let A2 = multiply(A, A);
        let A3 = multiply(A2, A);

        let c1 = trace;

        let principalMinors =
            (A[0][0]*A[1][1] - A[0][1]*A[1][0]) +
            (A[0][0]*A[2][2] - A[0][2]*A[2][0]) +
            (A[1][1]*A[2][2] - A[1][2]*A[2][1]);

        let polynomial = subtract(
            subtract(
                addMatrices(
                    A3,
                    scalarMultiply(A2, -c1)
                ),
                scalarMultiply(A, -principalMinors)
            ),
            scalarMultiply(identity(3), -det)
        );

        result = polynomial;
    }

    let valid = result.every(row =>
        row.every(value => Math.abs(value) < 0.000001)
    );

    document.getElementById("result").innerHTML =
        valid
        ? "✅ Cayley-Hamilton Theorem Verified!"
        : "❌ Verification failed. Check the matrix values.";
}

function addMatrices(A, B) {
    return A.map((row, i) =>
        row.map((value, j) => value + B[i][j])
    );
}

function clearMatrix() {
    createMatrix();
    document.getElementById("result").innerHTML = "";
}

createMatrix();

In [ ]:
Cayley-Hamilton-Verifier/
│
├── index.html
├── style.css
└── script.js